# 03 · Change detection over a time-series site

SAR's superpower is *repeat* imaging: the same radar geometry over the same
spot, days or weeks apart, sees through cloud and dark. Composite two passes
into one color image and change jumps out.

This notebook needs the `viz` extra:

```bash
pip install "umbra-py[viz]"
```

> *Contains Umbra open data, licensed under CC BY 4.0.*

## 1 · Find a site imaged more than once

An Umbra *task* is repeat imaging of one site, so any task with 2+ passes
gives us a co-registerable pair. We sample a modest window and group by task
until we find one — exactly the pattern the library's own live tests use.

In [ ]:
from umbra_py import UmbraCatalog

by_task: dict[str, list] = {}
for it in UmbraCatalog().search(start="2024-01-01", end="2024-12-31", limit=12):
    if "GEC" in it.available_assets and it.task:
        by_task.setdefault(it.task, []).append(it)

series = next((v for v in by_task.values() if len(v) >= 2), None)
assert series is not None, "no task with 2+ GEC passes in the sampled window"
print(f"{series[0].task}: {len(series)} passes")

## 2 · Pick two frames, evenly spaced in time

`select_change_frames` sorts the passes and picks `frames` of them spread from
earliest to latest — and keeps the comparison apples-to-apples (same
polarization). Two frames feed the RGB change composite.

In [ ]:
from umbra_py import select_change_frames

pair = select_change_frames(series, frames=2)
assert len(pair) == 2
for p in pair:
    print(p.datetime, "·", "/".join(p.polarizations) or "?")

## 3 · Composite the change

`save_change_composite` warps both scenes onto a common grid and renders the
difference in color. The semantics are worth memorizing:

- **green** = *appeared / brightened* in the later pass,
- **magenta** = *vanished / dimmed*,
- **grey** = unchanged.

So a new building or a filled reservoir reads green; a demolished structure or
a drained field reads magenta.

In [ ]:
from IPython.display import Image

from umbra_py import save_change_composite

png = save_change_composite(pair, "change.png", max_size=512)
assert png.exists()
assert png.read_bytes()[:8] == b"\x89PNG\r\n\x1a\n"

Image(filename=str(png))

## Where next

- **`umbra change --narrate`** (the `ai` extra) hands this composite *and* a
  deterministic per-block dB grid to a vision model and returns a plain-language
  report of what changed, where, and by how much — grounded in the numbers, not
  vibes.
- **`umbra watch`** turns this into a standing monitor: run the same search on a
  schedule and act only on the passes that are *new*.
- **`umbra timescan`** summarizes a whole series (not just two passes) into one
  "where did activity happen" image.

*Contains Umbra open data, licensed under CC BY 4.0.*